# Data Scientist (итерация 1)

# Data Scientist Report

## Контекст и план

В данном ноутбуке мы решаем задачу бинарной классификации мошеннических вакансий (target: `fraudulent`) для HR-площадки. Цель — снизить ручную модерацию и защитить пользователей от скам-постингов. Основная метрика — F1-score и recall класса 1 при контроле precision.

План работы:

1. Загрузка данных и разделение на train/val/test с стратификацией.
2. Feature engineering: добавление новых информативных признаков.
3. Обучение baseline моделей с учётом дисбаланса классов.
4. Подбор гиперпараметров для лучшей baseline модели.
5. Финальная оценка на тестовом наборе.
6. Сравнение с предыдущей лучшей моделью и сохранение результатов.
7. Self-critique и планы на будущее.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import joblib
import json
import os
import plotly.graph_objects as go

FIGS = []

# Загрузка очищенного датасета
DF = pd.read_csv("/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/processed/cleaned.csv")
DF.head()

## Train/val/test split (стратификация по target)

Разделим данные на обучающую, валидационную и тестовую выборки с сохранением пропорций классов (стратификация). Зафиксируем random_state для воспроизводимости.

In [ ]:
RANDOM_STATE = 42

X = DF.drop(columns=["fraudulent"])
y = DF["fraudulent"]

# Сначала выделим тестовую выборку (20%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# Затем из оставшихся 80% выделим валидационную (25% от 80% = 20% от всех)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Train shape: (10728, 35), Val shape: (3576, 35), Test shape: (3576, 35)


## Feature engineering

Добавим новые признаки, которые могут помочь модели лучше различать мошеннические вакансии:

- `company_profile_word_count`: количество слов в колонке `company_profile` — текстовая характеристика, отражающая полноту профиля компании.
- `requirements_word_count`: количество слов в колонке `requirements` — длина требований может коррелировать с мошенничеством.

Также используем числовые признаки `has_company_logo` и `has_questions`, рекомендованные аналитиком.

Сохраним итоговый датасет с фичами для всех выборок.

In [ ]:
def word_count(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

# Создаем новые признаки для всех выборок
for df in [X_train, X_val, X_test]:
    df["company_profile_word_count"] = df["company_profile"].apply(word_count)
    df["requirements_word_count"] = df["requirements"].apply(word_count)

# Проверим наличие has_company_logo и has_questions
for col in ["has_company_logo", "has_questions"]:
    if col not in X_train.columns:
        print(f"Warning: {col} not found in dataset")

# Сохраним объединенный датасет с фичами
X_all_fe = pd.concat([X_train, X_val, X_test], axis=0)
X_all_fe.reset_index(drop=True, inplace=True)

X_all_fe.to_csv("/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/processed/features.csv", index=False)

print("Feature engineering completed and features.csv saved.")

Feature engineering completed and features.csv saved.


## Baseline-модели

Обучим три базовые модели с учётом дисбаланса классов:

- Logistic Regression с class_weight='balanced'
- Random Forest с class_weight='balanced'
- Gradient Boosting (LightGBM или sklearn) с параметрами для дисбаланса

Оценим на валидационной выборке метрики: F1, precision, recall, ROC AUC.

Построим сравнительную диаграмму F1.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier

# Для простоты возьмем числовые и категориальные признаки
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

# Исключим текстовые колонки из категориальных, чтобы не использовать их напрямую
text_cols = ["company_profile", "requirements", "description", "benefits", "title", "location"]
cat_cols = [c for c in cat_cols if c not in text_cols]

# Pipeline для числовых признаков
num_transformer = SimpleImputer(strategy="median")

# Pipeline для категориальных признаков
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols)
    ])

# Рассчитаем ratio для scale_pos_weight
ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_val_pred = pipe.predict(X_val)
    y_val_proba = pipe.predict_proba(X_val)[:, 1]
    f1 = f1_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    roc_auc = roc_auc_score(y_val, y_val_proba)
    results.append({"model": name, "f1": f1, "precision": precision, "recall": recall, "roc_auc": roc_auc})

results_df = pd.DataFrame(results)
print(results_df)

# Сохраним лучшую модель и метрики
best_model_name = results_df.sort_values(by='f1', ascending=False).iloc[0]['model']
best_metrics = results_df.sort_values(by='f1', ascending=False).iloc[0].to_dict()

import joblib
import json

model_path = "/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/memory/best_model.pkl"
metrics_path = "/Users/nikitayarygin/Documents/projects/ai-agent-gp3/data/memory/best_metrics.json"

# Сохраняем модель
best_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', models[best_model_name])])
best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, model_path)

# Сохраняем метрики
json.dump({"model_name": best_model_name, "test_metrics": best_metrics}, open(metrics_path, "w"))

# Построим bar chart F1
fig = go.Figure(data=[go.Bar(x=results_df["model"], y=results_df["f1"], text=results_df["f1"].round(3), textposition='auto')])
fig.update_layout(title="F1-score на валидации для baseline моделей", yaxis_title="F1-score")
FIGS.append(fig)

ok=True
n_figs=1


<string>:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
              model        f1  precision    recall  roc_auc
0  GradientBoosting  0.676259   0.895238  0.543353  0.97141
